# Autonomous MOF synthesis — staged Bayesian optimisation (60-min protocol)

One notebook for the whole campaign: synthesis (`Experiment.make_batch`, as in `mof_synthesis.ipynb`), UV–Vis
scoring (Gualtieri fit, as in `gualtieri.ipynb`) and batch selection (GP + EI, as in `bayesian.ipynb`) run in a loop.

| Level | Vials per round | Reading interval | Readings | Last reading | Formulations |
|---|---|---|---|---|---|
| A (round 1) | 5 | 15 min | 5 | 60 min | 5 maximin seeds (`sampling.ipynb`) |
| A | 5 | 15 min | 5 | 60 min | GP + EI, k = 5 |
| B | 2 | 5 min | 13 | 60 min | GP + EI, k = 2 |
| C | 1 | 2 min | 13 | 24 min | GP + EI, k = 1 |

**Rule after every round:** α(t₁) = (I₄₀₀(t₁) − min I₄₀₀) / I_max of the Gualtieri fit, at the 2nd reading of each fitted vial.
If any vial has α(t₁) > 0.5 the next round moves one level down the table (A → B → C); otherwise it stays.
Levels never go back up. **The campaign stops after 4 rounds.**

Objective: k_n = 1/a from the 400-nm signal. **ZIF-8 set-up:** only the 400-nm gate is used (a vial whose 400-nm
absorbance never reaches 0.12 or rises by less than 0.12 scores 0); the 590-nm gate of ZIF-67 is off (`FEATURE_GATE_NM = None`).

**Between rounds** (no operator): every round starts with no lid on the deck (any lid is moved back onto the slot-4
stack), puts the sample and solvent lids on after its last reading (levels A and B also cover between readings when
the next reading is ≥150 s away, so that taking the lids off again never delays it; level C keeps them off for its
24 min of 2-min readings and covers at the end), and the precursor lid goes on once dispensing is done. The **dual syringe keeps its precursor solutions** from one round to the
next and is emptied back into the reservoirs only after the last round; the **single syringe is emptied every round**
(it is needed empty for mixing). The **dark/white references are recorded once**, in round 1, and reused afterwards
(also after a kernel restart).

The run stops by itself only when the operator is needed: a reservoir would run low (`paused_refill_reagents`),
or a hardware error before all readings were taken (`paused_hardware_error`). Round 1 asks you to switch the probe light
off/on for the dark/white references (two key presses); loading the rack offsets asks "1 or 2" as in `mof_synthesis.ipynb`.

Public entry point derived from the recorded ZIF-8 notebook. Output paths are
relative, saved cell output is cleared, and operator refill accounting is disabled
until explicitly requested. Scientific settings and controller modules are retained.
The original executed notebook is archived in `Dataset/ZIF8_20261104/`.
See `README_autonomous_run.md` before using laboratory hardware.


## Before you start (deck)

1. **Lids:** all 4 lids stacked in **slot 4**; no lid on slots 1, 2, 3, 5. The code starts every round by moving any
   lid back to slot 4, so the stack is always in a known state.
2. **Vials:** an empty vial in every position A1–A5, B1–B5 of the racks in **slot 2 and slot 5** (a 4-round campaign uses
   at most 20 positions; rounds fill slot 2 first, a round never spans two racks).
3. **Reservoirs:** fill the metal precursor (Zn²⁺), 2-mIm and TEA/methanol and enter the volumes in `REAGENTS_ML` below. Wash solvent in slot 3 well 2.
4. Single and dual syringes reset (empty) by hand; tools parked; labware removed from slots 0–1 before homing.
   The offsets are written by the "Manual Offsetting" cell below, with the same points as `mof_synthesis.ipynb`.

## 1. Imports

In [ ]:
import sys, os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use the science_jubilee package in Code/src (first on the path, so the updated files are the ones imported)
CODE_DIR = Path.cwd().resolve()
if not (CODE_DIR / "src").is_dir():
    CODE_DIR = CODE_DIR / "Code"
if not (CODE_DIR / "src").is_dir():
    raise FileNotFoundError("Open this notebook from the repository root or Code directory.")
sys.path.insert(0, str(CODE_DIR / "src"))

from science_jubilee.optimization.reaction_aware import (ReactionAwareBOLoop, Phase, campaign_initial_sampling,
                                                         PHASES_60MIN)
%matplotlib inline
# (offline_demo switches matplotlib to the non-interactive Agg backend on import; this brings inline plots back)
print("✓ imports")

## 2. Configuration

`LADDER` is the table above. Each `Phase` gives vials per round, interval (min), number of readings, and which level
follows a round with α(t₁) above (`next_if_fast`) or not above (`next_if_slow`) the threshold.
Do not change `LADDER` when resuming a run (the saved state checks it).

In [ ]:
OPERATOR_NAME = "Zuyang"
JUBILEE_IP = "192.168.1.2"
SPECTRUM_BASE_DIR = str(CODE_DIR.parent / "spectrum_data")
OUTPUT_DIR = str(CODE_DIR / "optimization_results" / "zif8_new_run")  # new folder per campaign

# Chemistry: ZIF-8 (Zn2+ / 2-mIm) has no visible band, so only the 400-nm signal is scored.
METAL_PRECURSOR_NAME = "Zn+2"
ORGANIC_PRECURSOR_NAME = "2-methylimidazole"
SOLVENT_NAME = "triethylamine/methanol"
FEATURE_GATE_NM = None       # second optical gate: None for ZIF-8; 590.0 for ZIF-67 (Co(II) band)
SIGNAL_GATE = 0.12           # 400-nm gate (and 590-nm gate if used): absorbance must reach and rise by this much

N_ROUNDS = 4                 # stop after this many measured rounds
ALPHA_THRESHOLD = 0.5        # alpha(t1) above which the next round moves one level down
SEED = 42                    # maximin seeds and GP restarts, as in sampling.ipynb / bayesian.ipynb

# Round-1 formulations: None = the 5 maximin seeds of sampling.ipynb,
# or your own 5 x [metal, 2-mIm, TEA] in mL (multiples of 0.1 mL, each row summing to 8.0 mL)
INITIAL_CONDITIONS = None

# mL now in each reservoir. The run pauses before a round would leave less than REAGENT_MARGIN_ML
# (one syringe refill draws a full syringe, ~10.7 mL). Set REAGENTS_ML = None to switch the check off.
REAGENTS_ML = {"metal": 40.0, "organic": 40.0, "solvent": 40.0}
REAGENT_MARGIN_ML = 12.0

LADDER = {
    0: Phase("seed_5vials_15min", batch_size=5, interval_min=15,  n_records=5,  seeded=True, next_if_fast=2, next_if_slow=1),
    1: Phase("bo_5vials_15min",   batch_size=5, interval_min=15,  n_records=5,  next_if_fast=2, next_if_slow=1),
    2: Phase("bo_2vials_5min",    batch_size=2, interval_min=5,   n_records=13, next_if_fast=3, next_if_slow=2),
    3: Phase("bo_1vial_2min",     batch_size=1, interval_min=2,   n_records=13, next_if_fast=3, next_if_slow=3,
             lids_between_readings=False),
}

RESUME = False               # True to continue a paused/interrupted run in OUTPUT_DIR

for p, ph in LADDER.items():
    print(f"level {p} {ph.name:18s}: {ph.batch_size} vials, every {ph.interval_min} min x {ph.n_records} readings "
          f"(0-{ph.last_reading_min:g} min); alpha(t1)>{ALPHA_THRESHOLD} -> {ph.next_if_fast}, else -> {ph.next_if_slow}")

## 3. Preview (no hardware)

In [ ]:
X0 = (np.asarray(INITIAL_CONDITIONS, float) if INITIAL_CONDITIONS is not None
      else campaign_initial_sampling(LADDER[0].batch_size, random_state=SEED))
print(f"Round 1 formulations ({METAL_PRECURSOR_NAME} / 2-mIm / TEA, mL):")
for i, x in enumerate(X0, 1):
    print(f"  {i}: {x[0]:.1f} / {x[1]:.1f} / {x[2]:.1f}")
need = X0.sum(axis=0)
print(f"Round 1 uses {METAL_PRECURSOR_NAME} {need[0]:.1f} mL, 2-mIm {need[1]:.1f} mL, TEA {need[2]:.1f} mL")
print(f"Worst case over {N_ROUNDS} rounds: {LADDER[0].batch_size + (N_ROUNDS - 1) * max(ph.batch_size for ph in LADDER.values())} vials "
      f"(20 positions on the two racks)")

## 4. Hardware set-up (same as `mof_synthesis.ipynb`)

Manual offsets — the cell below is the "Manual Offsetting" cell of `mof_synthesis.ipynb`, unchanged (same points);
it writes the offsets that the labware cell loads.

In [ ]:
from science_jubilee.decks.Deck import Deck

# Manual Offsetting

deck = Deck("lab_automation_deck") # We are not using the deck, so operate any deck.json 

solvents = deck.load_labware("uwsunlab_2_wellplate_60000ul_slot3.json", slot = 3, has_lid_on_top= False, currentLiquidVolume= 60)
solvents.manual_offset_small_plate([[252.0, 155.8], [195.0, 155.8]], save = True)

# -------------Position JSON file for Single Syringe ---------------------#
samples2_ssy = deck.load_labware("uwsunlab_10_wellplate_14000ul_ssy_1.json", slot = 2, has_lid_on_top= False) 
samples2_ssy.manual_offset2([[132.0, 128], [38.0, 128], [38.0, 169.0]], save = True)
samples5_ssy = deck.load_labware("uwsunlab_10_wellplate_14000ul_ssy_2.json", slot = 5, has_lid_on_top= False)
samples5_ssy.manual_offset2([[270.0, 225.0], [177.0, 225.0], [177.0, 266.0]], save = True)
    
# -------------Position JSON file for Dual Syringe ---------------#
samples2_sy = deck.load_labware('uwsunlab_10_wellplate_14000ul_sy_1.json', slot = 2, has_lid_on_top= False)       #new config
samples2_sy.manual_offset2([[131.0, 128.0],[36.7, 128.0], [36.7, 169.5]], save = True)
samples5_sy = deck.load_labware('uwsunlab_10_wellplate_14000ul_sy_2.json', slot = 5, has_lid_on_top= False)     #new config
samples5_sy.manual_offset2([[271.0, 225.0], [177.0, 225.0], [177.0, 266.0]], save = True)

precursors = deck.load_labware('uwsunlab_2_wellplate_60000ul_slot1.json', slot = 1, has_lid_on_top= False, currentLiquidVolume= 60)
precursors.manual_offset_small_plate([[249.0, 52.0], [195.0, 52.0]], save = True)

# -----------Position JSON file for Spectrometer----------------# 
samples2_spec = deck.load_labware('uwsunlab_10_wellplate_14000ul_spec_1.json', slot = 2, has_lid_on_top= False)   #new config
samples2_spec.manual_offset2([[125.5, 126],[34.5, 126], [34.5, 167]], save = True)
samples5_spec = deck.load_labware('uwsunlab_10_wellplate_14000ul_spec_2.json', slot = 5, has_lid_on_top= False)   #new config
samples5_spec.manual_offset2([[266, 224], [174, 224], [174, 265]], save = True)

⚠️ Remove labware from slots 0 and 1 before homing.

In [ ]:
from science_jubilee.Machine import Machine
from science_jubilee.Experiment import Experiment
from science_jubilee.tools.Double_Syringe import DoubleSyringe
from science_jubilee.tools.Syringe import Syringe
from science_jubilee.tools.Vacuum_Gripper import VacuumGripper
from science_jubilee.tools.Oceandirect_axo import Spectrometer

axo = Machine(address=JUBILEE_IP)

In [ ]:
axo.home_all()
axo.move_to(z=200)  # Safest Z location

In [ ]:
# Loading the labwares (same as mof_synthesis.ipynb; each 10-vial rack asks "1 or 2" for its 3-point offset)
deck = axo.load_deck("lab_automation_deck")

solvents = axo.load_labware('uwsunlab_2_wellplate_60000ul_slot3.json', slot=3, has_lid_on_top=False, currentLiquidVolume=60)
solvents.load_manualOffset()

## ---------------------Single Syringe Pump ----------------------------##
samples2_ssy = axo.load_labware('uwsunlab_10_wellplate_14000ul_ssy_1.json', slot=2, has_lid_on_top=False)
samples2_ssy.load_manualOffset()
samples5_ssy = axo.load_labware('uwsunlab_10_wellplate_14000ul_ssy_2.json', slot=5, has_lid_on_top=False)
samples5_ssy.load_manualOffset()

## ---------------------Dual Syringe Pump ----------------------------##
samples2_sy = axo.load_labware('uwsunlab_10_wellplate_14000ul_sy_1.json', slot=2, has_lid_on_top=False)
samples2_sy.load_manualOffset()
samples5_sy = axo.load_labware('uwsunlab_10_wellplate_14000ul_sy_2.json', slot=5, has_lid_on_top=False)
samples5_sy.load_manualOffset()

precursors = axo.load_labware('uwsunlab_2_wellplate_60000ul_slot1.json', slot=1, has_lid_on_top=False, currentLiquidVolume=60)
precursors.load_manualOffset()

## --------------------------Spectrometer -------------------------------##
samples2_spec = axo.load_labware('uwsunlab_10_wellplate_14000ul_spec_1.json', slot=2, has_lid_on_top=False)
samples2_spec.load_manualOffset()
samples5_spec = axo.load_labware('uwsunlab_10_wellplate_14000ul_spec_2.json', slot=5, has_lid_on_top=False)
samples5_spec.load_manualOffset()

## ------------------------Vacuum Gripper --------------------------------##
vacuum_location = {0: {"loc": (84, 51, 0), "labwares_list": []},                                             # Slot 0
                   1: {"loc": (225, 57, 0), "labwares_list": [precursors]},                                  # Slot 1 (2 Precursor Container)
                   2: {"loc": (84, 147, 0), "labwares_list": [samples2_ssy, samples2_sy, samples2_spec]},    # Slot 2 (10 Well Plate)
                   3: {"loc": (225, 149, 0), "labwares_list": [solvents]},                                   # Slot 3 (Solvent & Cleansing liquid Container)
                   4: {"loc": (79, 244, 0), "labwares_list": []},                                            # Slot 4 (Lid stack)
                   5: {"loc": (221, 244, 0), "labwares_list": [samples5_ssy, samples5_sy, samples5_spec]}}   # Slot 5 (10 Well Plate)

In [ ]:
# Tools
single_syringe = Syringe(index=0, name='single_syringe', config='single_syringe')
axo.load_tool(single_syringe)

dual_syringe = DoubleSyringe(index=2, name='Dual_Syringe', config='10cc_syringe')
axo.load_tool(dual_syringe)

gripper = VacuumGripper(index=4, name='Vacuum_Gripper', vacuum_pin=0, limit_switch_pin=2)
axo.load_tool(gripper)

# The loop gives every round its own data folder under SPECTRUM_BASE_DIR/<date>/<operator>/
spectrometer = Spectrometer(index=3,
                            name='Spectrometer',
                            base_dir=SPECTRUM_BASE_DIR,
                            experiment_name='autonomous_mof_synthesis',
                            operator_name=OPERATOR_NAME,
                            target_compound='ZIF-8',
                            experiment_notes=f'Staged BO, 60-min protocol, {N_ROUNDS} rounds',
                            solvent='triethylamine/methanol',
                            temperature_c=25)
axo.load_tool(spectrometer)

Caution (from `mof_synthesis.ipynb`):
1. Make sure all tools are in their parking positions — do a pick-up/park test of every tool first.
2. All labware (precursors, solvents, both vial racks) in the correct slot; **4 lids stacked in slot 4**.
3. Single and dual syringes manually reset before the experiment.

In [ ]:
all_labwares = {"slot1": {"precursors": precursors},
                "slot2": {"samples2_ssy": samples2_ssy, "samples2_sy": samples2_sy, "samples2_spec": samples2_spec},
                "slot3": {"solvents": solvents},
                "slot4": {"vacuum_location": vacuum_location},
                "slot5": {"samples5_ssy": samples5_ssy, "samples5_sy": samples5_sy, "samples5_spec": samples5_spec}}
all_tools = {"single_syringe": single_syringe, "dual_syringe": dual_syringe, "spectrometer": spectrometer, "gripper": gripper}

exp = Experiment(machine=axo, deck=deck, all_tools=all_tools, all_labwares=all_labwares)
print("✓ Experiment ready")

## 5. Create (or resume) the loop

In [ ]:
loop = ReactionAwareBOLoop(
    experiment=exp,
    output_dir=OUTPUT_DIR,
    operator_name=OPERATOR_NAME,
    phases=LADDER,
    max_total_batches=N_ROUNDS,
    alpha_threshold=ALPHA_THRESHOLD,
    initial_conditions=INITIAL_CONDITIONS,
    reagents_ml=REAGENTS_ML,
    reagent_margin_ml=REAGENT_MARGIN_ML,
    pause_on_phase_change=False,     # change level without stopping
    keep_dual_syringe_between_batches=True,   # empty the dual syringe only after the last round
    feature_gate_nm=FEATURE_GATE_NM,          # None: score on the 400-nm signal only (ZIF-8)
    signal_gate=SIGNAL_GATE,
    reagent_names={"metal": METAL_PRECURSOR_NAME, "organic": ORGANIC_PRECURSOR_NAME, "solvent": SOLVENT_NAME},
    seed=SEED,
)

if loop.state_file.exists():
    if not RESUME:
        raise RuntimeError(f"{loop.state_file} exists: set RESUME = True to continue that run, or choose a new OUTPUT_DIR.")
    loop.load_state()
    print(f"Resumed: round {loop._n_measured()} done, level {loop.phase} ({loop.phases[loop.phase].name}), "
          f"status {loop.status}, reagents left {loop.reagents}")
else:
    print(f"New campaign in {loop.output_dir}")

## 6. Run

Runs all rounds. The first round asks you to switch the probe light **off** (dark) and **on** (white).
If the cell stops with a pause, follow the message, then re-run this cell.

In [ ]:
result = loop.run()

print("=" * 70)
print(result["status"].replace("_", " ").upper())
print(result["message"])
print(f"Rounds measured: {result['batches_measured']} / {N_ROUNDS}; current level {result['phase']} ({result['phase_name']})")
print(f"Reagents left (mL, estimated): {result['reagents_left_ml']}")
if result["best_k_n"] is not None:
    print(f"Best k_n so far: {result['best_k_n']:.4f} min^-1 at {METAL_PRECURSOR_NAME}/2-mIm/TEA = {result['best_composition_mL']} mL")

### If the run paused

* **`paused_refill_reagents`** – refill the reservoir(s) named in the message, enter the volumes now in the reservoirs
  below, run that cell, then re-run the *Run* cell.
* **`paused_hardware_error`** – an exception stopped `make_batch` before all readings were taken (message shows it).
  Fix the deck, set `DISCARD = True` below and run that cell (the round's vial positions stay marked as used), then re-run *Run*.
  A round that crashes **after** its last reading is analysed automatically.
* **`paused_analysis_failed`** – data folder incomplete or recipe mismatch; check the folder in the message.
* **Stopping / restarting the kernel in the middle of a campaign** – first run the "Empty the dual syringe" cell in
  section 8 (the syringe otherwise still holds precursor solution), then after the restart move every lid back onto
  the stack in slot 4 (the code assumes no lids on the deck after a restart) and set `RESUME = True`.
  The dark/white references of round 1 are reused; to record new ones set `exp.need_spectrum_refs = True` before *Run*.

In [ ]:
# Run only after an actual refill or a deliberate decision to discard a pending batch.
REFILL_REAGENTS_ML = None  # Example after a physical refill: {"metal": 60.0, "organic": 60.0, "solvent": 60.0}
if REFILL_REAGENTS_ML is not None:
    loop.refill_reagents(**REFILL_REAGENTS_ML)

DISCARD = False
if DISCARD:
    loop.discard_pending()
